<a href="https://colab.research.google.com/github/contatowillian/fiap/blob/desafio_3/8IADT_Fase_3_Tech_challenge_com_ppt_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Projeto Tech Challenge - Assistente Virtual Médico Personalizado

Este notebook contém o passo a passo completo e modularizado para o desenvolvimento de um Assistente Virtual Médico Personalizado, utilizando Fine-Tuning de LLM, LangChain/LangGraph e mecanismos de segurança, conforme os requisitos do Tech Challenge - Fase 3.

## 1. Instalação de Bibliotecas Essenciais

Vamos começar instalando todas as bibliotecas necessárias para o projeto. Isso inclui ferramentas para fine-tuning (Unsloth, Peft, TRL), manipulação de LLMs (transformers, accelerate, bitsandbytes), e construção de agentes (LangChain, LangGraph, datasets).

In [1]:
# Instalação de bibliotecas necessárias para fine-tuning, LangChain e LangGraph.
# A flag -qq (quiet) é usada para instalações mais limpas no Colab, exceto onde precisamos de mais detalhes.

# 1. Atualizar pip
!pip install -qqq --upgrade pip

# 2. **LIMPEZA AGRESSIVA DE DIRETÓRIOS LANGCHAIN RESIDUAIS**
# Isso é feito para garantir que não haja arquivos antigos ou corrompidos de instalações anteriores.
import os
import shutil
import site

site_packages_path = site.getsitepackages()[0]
print(f"\nRealizando limpeza agressiva de diretórios LangChain residuais em: {site_packages_path}")

langchain_prefixes = [
    'langchain',
    'langchain_community',
    'langchain_core',
    'langchain_text_splitters',
    'langgraph',
    'langgraph_prebuilt',
    'langgraph_sdk',
    'langchain_classic',
    'langsmith'
]

for item_name in os.listdir(site_packages_path):
    for prefix in langchain_prefixes:
        if item_name.startswith(prefix):
            full_path = os.path.join(site_packages_path, item_name)
            try:
                if os.path.isdir(full_path):
                    print(f"Removendo diretório: {full_path}")
                    shutil.rmtree(full_path)
                elif os.path.isfile(full_path) and (item_name.endswith('.py') or item_name.endswith('.egg-link')):
                    print(f"Removendo arquivo: {full_path}")
                    os.remove(full_path)
            except Exception as e:
                print(f"Erro ao remover {full_path}: {e}")
                pass # Continue tentando remover outros

print("Limpeza concluída. Prosseguindo com a instalação limpa.\n")

# 3. Instalar `typing-extensions` e `packaging` explicitamente e cedo.
#    Usamos `--ignore-installed` para evitar conflitos com versões do sistema que não podem ser desinstaladas.
print("Reinstalando typing-extensions e packaging com tratamento de conflitos...")
!pip install -qqq --upgrade typing-extensions packaging --ignore-installed

# 4. Instalar Unsloth e suas dependências otimizadas para Colab.
#    Esta é a instalação mais crítica, e deve ser feita antes de outras que podem ter conflitos.
#    Unsloth trará Pydantic v2 e pydantic-core, e suas próprias versões de transformers, accelerate, bitsandbytes, numpy, pandas.
print("Instalando Unsloth e suas dependências otimizadas...")
!pip install -qqq --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 5. Instalar outras dependências de Hugging Face não incluídas diretamente no Unsloth.
#    Essas geralmente são menos propensas a grandes conflitos.
print("Instalando dependências adicionais do Hugging Face (peft, trl, datasets)...")
!pip install -qqq peft trl datasets

# 6. Instalar nltk e sentence-transformers explicitamente (para resolver o erro de nltk e embeddings).
print("Instalando nltk e sentence-transformers...")
!pip install -qqq nltk sentence-transformers

# 7. Ferramentas para LangChain e LangGraph.
print("Desinstalando todos os componentes do LangChain para uma reinstalação limpa...")
# Desinstalação agressiva para garantir que nenhuma versão antiga ou corrompida persista.
!pip uninstall -y langchain langchain_community langchain_core langgraph langchain-text-splitters langsmith langchain-classic langgraph-prebuilt langgraph-sdk

print("Instalando bibliotecas LangChain e LangGraph com versões recentes e forçadas...")
# Reinstalamos com --force-reinstall e --no-cache-dir para evitar o uso de pacotes corrompidos do cache.
!pip install --force-reinstall --no-cache-dir langchain langchain_community langchain_core langgraph langchain-text-splitters

# 8. Para visualização (opcional) e PDF
print("Instalando bibliotecas de visualização e PDF (matplotlib, seaborn, pypdf)...")
!pip install -qqq matplotlib seaborn pypdf

print("Instalação de todas as bibliotecas concluída!")

print("\nVerificando versões instaladas de pacotes chave:")
!pip show packaging
!pip show numpy
!pip show pandas
!pip show pydantic
!pip show pydantic-core
!pip show typing-extensions
!pip show nltk
!pip show sentence-transformers
!pip show langchain
!pip show langchain_community
!pip show langchain_core
!pip show langchain-text-splitters
!pip show langgraph
!pip show langsmith
!pip show langchain-classic
!pip show langgraph-prebuilt
!pip show langgraph-sdk
!pip show unsloth


Realizando limpeza agressiva de diretórios LangChain residuais em: /usr/local/lib/python3.13/dist-packages
Removendo diretório: /usr/local/lib/python3.13/dist-packages/langgraph-1.2.11.dist-info
Removendo diretório: /usr/local/lib/python3.13/dist-packages/langchain-1.4.0.dist-info
Removendo diretório: /usr/local/lib/python3.13/dist-packages/langsmith
Removendo diretório: /usr/local/lib/python3.13/dist-packages/langchain_text_splitters-1.1.2.dist-info
Removendo diretório: /usr/local/lib/python3.13/dist-packages/langchain_classic-1.0.8.dist-info
Removendo diretório: /usr/local/lib/python3.13/dist-packages/langchain_community
Removendo diretório: /usr/local/lib/python3.13/dist-packages/langgraph_prebuilt-1.1.0.dist-info
Removendo diretório: /usr/local/lib/python3.13/dist-packages/langgraph_sdk-0.4.4.dist-info
Removendo diretório: /usr/local/lib/python3.13/dist-packages/langchain_text_splitters
Removendo diretório: /usr/local/lib/python3.13/dist-packages/langchain_protocol-0.0.19.dist-inf

Instalando bibliotecas de visualização e PDF (matplotlib, seaborn, pypdf)...
Instalação de todas as bibliotecas concluída!

Verificando versões instaladas de pacotes chave:
Name: packaging
Version: 26.3
Summary: Core utilities for Python packages
Home-page: 
Author: 
Author-email: Donald Stufft <donald@stufft.io>
License-Expression: Apache-2.0 OR BSD-2-Clause
Location: /usr/local/lib/python3.13/dist-packages
Requires: 
Required-by: accelerate, altair, arviz, astropy, bigquery-magics, bitsandbytes, bokeh, cudf-cu12, cudf-polars-cu12, cuml-cu12, dask, dataproc-spark-connect, datasets, db-dtypes, deprecation, distributed, faiss-cpu, fastai, geopandas, google-adk, google-cloud-bigquery, gradio, gradio_client, h5netcdf, huggingface_hub, ipykernel, jupyter-events, jupyter_server, jupytext, kaggle, kagglehub, keras, keras-hub, langchain-core, langsmith, lazy-loader, libpysal, matplotlib, momepy, nbconvert, nibabel, numba-cuda, pandas-gbq, panel, panel-material-ui, patsy, peft, plotly, pooch, 

In [2]:
try:
    import langchain
    import langchain_community
    import langchain_core
    import langchain_text_splitters
    print("LangChain e módulos relacionados importados com sucesso!")
except ImportError as e:
    print(f"Erro ao importar módulos do LangChain: {e}")
    print("Pode ser necessário reiniciar o ambiente de execução (Runtime -> Restart runtime) e tentar novamente.")

/tmp/ipykernel_17862/1390487792.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


LangChain e módulos relacionados importados com sucesso!


In [3]:
import os
import site

try:
    import langchain
    print(f"Pacote 'langchain' importado de: {langchain.__path__[0]}")

    # Tenta importar funções específicas de langchain.chains para verificar sua existência
    from langchain.chains.combine_documents import create_stuff_documents_chain
    from langchain.chains.retrieval import create_retrieval_chain
    print("Funções 'create_stuff_documents_chain' e 'create_retrieval_chain' importadas com sucesso!")

except ImportError as e:
    print(f"Erro ao importar módulos de 'langchain.chains': {e}")
    print("Isso pode indicar que o ambiente de execução precisa ser reiniciado ou que há um problema na instalação da versão correta do LangChain.")
except Exception as e:
    print(f"Ocorreu um erro inesperado ao inspecionar o pacote LangChain: {e}")

Pacote 'langchain' importado de: /usr/local/lib/python3.13/dist-packages/langchain
Erro ao importar módulos de 'langchain.chains': No module named 'langchain.chains'
Isso pode indicar que o ambiente de execução precisa ser reiniciado ou que há um problema na instalação da versão correta do LangChain.


In [4]:
# Tentativa de instalação do Unsloth conforme solicitado pelo usuário.
# Lembre-se: uma GPU ativa é crucial para que esta instalação seja bem-sucedida.
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print("Comando de instalação do Unsloth executado. Verifique os logs acima para confirmação de sucesso.")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-6wxhmn6v/unsloth_c8c043f3405245ab87e61afc212c8673
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-6wxhmn6v/unsloth_c8c043f3405245ab87e61afc212c8673
  Resolved https://github.com/unslothai/unsloth.git to commit a32099207f8e185101353e606a26cc0f14249409
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Comando de instalação do Unsloth executado. Verifique os logs acima para confirmação de sucesso.


In [5]:
# Instalação de bibliotecas necessárias para fine-tuning, LangChain e LangGraph.
# A flag -qq (quiet) é usada para instalações mais limpas no Colab.

# 1. Atualizar pip
!pip install -qqq --upgrade pip

# 2. Instalar `typing-extensions` explicitamente e cedo, pois muitas bibliotecas dependem dela
# Usamos `--upgrade --ignore-installed` para garantir a instalação de uma versão compatível
# sem tentar desinstalar uma versão do sistema que não pode ser removida.
print("Instalando/Atualizando typing-extensions para evitar conflitos...")
!pip install -qqq --upgrade typing-extensions --ignore-installed

# 3. Workaround para o erro 'Cannot uninstall packaging 24.0':
# A versão do sistema (packaging 24.0) não pode ser desinstalada. Muitos pacotes modernos,
# incluindo unsloth, requerem packaging >= 24.1. Vamos instalar uma versão compatível
# ignorando a versão do sistema com --ignore-installed. Isso fará com que o pip instale uma nova versão
# em '/usr/local/lib/python3.13/dist-packages' que será priorizada pelo Python.
print("Reinstalando packaging >= 24.1 ignorando a versão do sistema...")
!pip install -qqq --upgrade packaging>=24.1 --ignore-installed

# 4. Instalar Unsloth e suas dependências otimizadas para Colab
#    Unsloth trará Pydantic v2 e pydantic-core, evitando desinstalações conflitantes.
#    NÃO estamos atualizando numpy e pandas explicitamente aqui. Deixaremos Unsloth gerenciar suas próprias dependências para eles.
print("Instalando Unsloth e suas dependências...")
!pip install -qqq --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 5. Instalar outras dependências de Hugging Face não incluídas diretamente no Unsloth
print("Instalando dependências adicionais do Hugging Face...")
!pip install -qqq peft trl datasets

# 6. Ferramentas para LangChain e LangGraph (deixando o pip resolver as versões mais recentes compatíveis)
print("Instalando bibliotecas LangChain e LangGraph...")
# Para evitar conflitos de Pydantic v1 vs v2, vamos instalar versões mais recentes
# de LangChain que geralmente são compatíveis com Pydantic v2.
!pip install -qqq langchain langchain_community langchain_core langgraph
!pip install -qqq langchain-text-splitters

# 7. Para visualização (opcional) e PDF
print("Instalando bibliotecas de visualização e PDF...")
!pip install -qqq matplotlib seaborn pypdf

print("Instalação de todas as bibliotecas concluída!")

print("\nVerificando versões instaladas de pacotes chave:")
!pip show packaging
!pip show numpy
!pip show pandas
!pip show pydantic
!pip show pydantic-core
!pip show typing-extensions
!pip show langchain
!pip show langchain_community
!pip show langchain_core
!pip show langchain-text-splitters

Instalando/Atualizando typing-extensions para evitar conflitos...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires websockets<16,>=15.0.1, but you have websockets 16.1.1 which is incompatible.
Reinstalando packaging >= 24.1 ignorando a versão do sistema...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires websockets<16,>=15.0.1, but you have websockets 16.1.1 which is incompatible.
Instalando Unsloth e suas dependências...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Instalando dependências adicionais do Hugging Face...
Instalando bibliotecas LangChain e LangGraph...
Instalando bibliotecas de visuali

In [6]:
print("Instalando pydantic >= 2.0...")
!pip install -qqq pydantic>=2.0
print("Verificando a versão do pydantic instalada:")
!pip show pydantic

Instalando pydantic >= 2.0...
Verificando a versão do pydantic instalada:
Name: pydantic
Version: 2.13.5
Summary: Data validation using Python type hints
Home-page: https://github.com/pydantic/pydantic
Author: 
Author-email: Samuel Colvin <s@muelcolvin.com>, Eric Jolibois <em.jolibois@gmail.com>, Hasan Ramezani <hasan.r67@gmail.com>, Adrian Garcia Badaracco <1755071+adriangb@users.noreply.github.com>, Terrence Dorsey <terry@pydantic.dev>, David Montague <david@pydantic.dev>, Serge Matveenko <lig@countzero.co>, Marcelo Trylesinski <marcelotryle@gmail.com>, Sydney Runkle <sydneymarierunkle@gmail.com>, David Hewitt <mail@davidhewitt.dev>, Alex Hall <alex.mojaki@gmail.com>, Victorien Plot <contact@vctrn.dev>
License-Expression: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: annotated-types, pydantic-core, typing-extensions, typing-inspection
Required-by: albumentations, blosc2, fastapi, google-adk, google-genai, google-generativeai, gradio, langchain, langchain-classic, la

 ## 2. Configuração do Ambiente e GPU

É fundamental verificar se o ambiente do Google Colab está configurado corretamente para utilizar a GPU, o que é crucial para o fine-tuning de LLMs. Também faremos as importações iniciais das bibliotecas principais.

In [7]:
# Importa bibliotecas essenciais e verifica a disponibilidade da GPU.
import torch
import pandas as pd
import numpy as np
import logging
import os

# Verifica e imprime a disponibilidade da GPU
if torch.cuda.is_available():
    print(f"GPU disponível: {torch.cuda.get_device_name(0)}")
    print(f"Versão CUDA: {torch.version.cuda}")
    device = "cuda"
else:
    print("GPU não disponível. O treinamento pode ser muito lento ou inviável.")
    device = "cpu"

# Configurações básicas de logging para acompanhar o processo
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.info("Ambiente configurado com sucesso. GPU verificada.")

GPU disponível: Tesla T4
Versão CUDA: 12.8


 ## 3. FINE-TUNING DE LLM MÉDICA

Esta seção abordará a preparação dos dados médicos, o pré-processamento, a anonimização (simplificada para demonstração) e o fine-tuning de uma LLM open-source leve. Escolheremos o modelo Llama-3-8B-Instruct para o fine-tuning, utilizando Unsloth para otimização de recursos.

### 3.1 Preparação, Pré-processamento e Anonimização de Dados Médicos/Sintéticos

Para o fine-tuning, precisamos de um dataset de perguntas e respostas médicas. Como não temos acesso a dados reais de pacientes, vamos gerar um dataset sintético que simule o estilo de um dataset médico. Em um cenário real, você usaria datasets como PubMedQA, MedQuAD ou dados internos anonimizados.

**Nota sobre Anonimização:** A anonimização real de dados médicos é um processo complexo que envolve técnicas sofisticadas (HIPAA, LGPD) e geralmente requer ferramentas dedicadas e validação humana. Aqui, faremos uma demonstração simplificada para ilustrar o conceito.

In [8]:
# Gerando um dataset sintético para fine-tuning.
# Em um cenário real, você carregaria dados de um arquivo CSV, JSON ou de um dataset como o Hugging Face.

# Lista de exemplos de interações médico-paciente
medical_qa_data = [
    {"pergunta": "Quais são os sintomas comuns de uma gripe?", "resposta": "Joelho Doendo, febre, dor de cabeça, dores musculares, tosse, dor de garganta e fadiga.", "fonte": "Ministério da Saúde"},
    {"pergunta": "Como prevenir o diabetes tipo 2?", "resposta": "A prevenção do diabetes tipo 2 envolve manter uma dieta saudável, praticar exercícios regularmente, controlar o peso e evitar o consumo excessivo de açúcares.", "fonte": "Sociedade Brasileira de Diabetes"},
    {"pergunta": "Qual é o tratamento para hipertensão arterial?", "resposta": "O tratamento para hipertensão geralmente inclui mudanças no estilo de vida (dieta, exercício), e frequentemente medicamentos anti-hipertensivos prescritos por um médico.", "fonte": "Organização Mundial da Saúde"},
    {"pergunta": "Explique o que é vacinação.", "resposta": "Vacinação é a administração de uma substância (vacina) para estimular o sistema imunológico de uma pessoa a produzir anticorpos e desenvolver imunidade contra uma doença infecciosa específica.", "fonte": "Fiocruz"},
    {"pergunta": "Quais os benefícios de uma dieta rica em fibras?", "resposta": "Uma dieta rica em fibras ajuda na digestão, previne constipação, auxilia no controle do açúcar no sangue e pode reduzir o risco de doenças cardíacas.", "fonte": "Associação Americana do Coração"},
    {"pergunta": "Posso tomar paracetamol para dor de cabeça?", "resposta": "Sim, paracetamol é um analgésico comum para dor de cabeça, mas sempre siga a dosagem recomendada na bula ou a indicação do seu médico. Não exceda a dose diária máxima.", "fonte": "Bula do Medicamento"}
]

# Convertendo para um DataFrame Pandas
df_medical = pd.DataFrame(medical_qa_data)

# Exemplo de pré-processamento e 'anonimização' simplificada (apenas para demonstração conceitual)
def preprocess_and_anonymize(text):
    # Exemplo: remover menções a nomes genéricos ou números de identificação fictícios
    text = text.replace("paciente X", "paciente")
    text = text.replace("ID12345", "[ID_ANONIMIZADO]")
    # Em um cenário real, você usaria técnicas mais avançadas para PHI (Protected Health Information)
    return text

df_medical['pergunta_anon'] = df_medical['pergunta'].apply(preprocess_and_anonymize)
df_medical['resposta_anon'] = df_medical['resposta'].apply(preprocess_and_anonymize)

print("Dataset médico sintético criado e 'anonimizado' (simplificado).")
display(df_medical.head())

# Formato para fine-tuning (chat_template)
# Unsloth prefere um formato de lista de dicionários para o dataset
# Cada entrada deve ter uma chave 'messages' com uma lista de turnos de diálogo.

def format_data_for_unsloth(df):
    formatted_data = []
    for index, row in df.iterrows():
        formatted_data.append({
            "messages": [
                {"role": "user", "content": row['pergunta_anon']},
                {"role": "assistant", "content": row['resposta_anon']}
            ]
        })
    return formatted_data

formatted_medical_data = format_data_for_unsloth(df_medical)

# Exibe um exemplo do formato para fine-tuning
print("\nExemplo de dado formatado para Unsloth:")
print(formatted_medical_data[0])

from datasets import Dataset

# Converte a lista de dicionários para um Dataset do Hugging Face
dataset = Dataset.from_pandas(pd.DataFrame(formatted_medical_data))
print(f"\nDataset Hugging Face criado com {len(dataset)} exemplos.")

Dataset médico sintético criado e 'anonimizado' (simplificado).


,pergunta,resposta,fonte,pergunta_anon,resposta_anon
0,Quais são os sintomas comuns de uma gripe?,"Joelho Doendo, febre, dor de cabeça, dores mus...",Ministério da Saúde,Quais são os sintomas comuns de uma gripe?,"Joelho Doendo, febre, dor de cabeça, dores mus..."
1,Como prevenir o diabetes tipo 2?,A prevenção do diabetes tipo 2 envolve manter ...,Sociedade Brasileira de Diabetes,Como prevenir o diabetes tipo 2?,A prevenção do diabetes tipo 2 envolve manter ...
2,Qual é o tratamento para hipertensão arterial?,O tratamento para hipertensão geralmente inclu...,Organização Mundial da Saúde,Qual é o tratamento para hipertensão arterial?,O tratamento para hipertensão geralmente inclu...
3,Explique o que é vacinação.,Vacinação é a administração de uma substância ...,Fiocruz,Explique o que é vacinação.,Vacinação é a administração de uma substância ...
4,Quais os benefícios de uma dieta rica em fibras?,"Uma dieta rica em fibras ajuda na digestão, pr...",Associação Americana do Coração,Quais os benefícios de uma dieta rica em fibras?,"Uma dieta rica em fibras ajuda na digestão, pr..."



Exemplo de dado formatado para Unsloth:
{'messages': [{'role': 'user', 'content': 'Quais são os sintomas comuns de uma gripe?'}, {'role': 'assistant', 'content': 'Joelho Doendo, febre, dor de cabeça, dores musculares, tosse, dor de garganta e fadiga.'}]}

Dataset Hugging Face criado com 6 exemplos.


### 3.2 Treinamento/Fine-Tuning de uma LLM Open-Source Leve com Unsloth

Nesta subseção, vamos realizar o fine-tuning de uma LLM open-source leve, como o Llama-3-8B-Instruct, utilizando a biblioteca Unsloth. O Unsloth otimiza o uso de memória e velocidade de treinamento, tornando o fine-tuning de modelos grandes viável em GPUs como a T4 disponível no Google Colab (quando ativada).

Usaremos a técnica LoRA (Low-Rank Adaptation) para ajustar o modelo de forma eficiente, treinando apenas uma pequena parte dos pesos do modelo, o que reduz drasticamente os requisitos de computação e memória.

In [9]:
from unsloth import FastLanguageModel
import torch
import os

# Set PyTorch CUDA allocator configuration to handle fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

max_seq_length = 256 # Further reduced sequence length to save more memory
dtype = torch.float16 # Explicitly set to float16 for T4 compatibility
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-2-7b-bnb-4bit", # Changed to a smaller, T4-friendly model
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map = {"": 0}, # Explicitly map all layers to GPU 0
)

# --- NOVO CÓDIGO PARA CARREGAR CHAT_TEMPLATE ---
# Garante que o chat_template seja carregado para este tokenizer
lora_model_path = "./lora_adapters" # O caminho onde o chat_template está salvo
chat_template_path = os.path.join(lora_model_path, "chat_template.jinja")

if os.path.exists(chat_template_path):
    with open(chat_template_path, "r", encoding="utf-8") as f:
        tokenizer.chat_template = f.read()
    print(f"Chat template carregado de '{chat_template_path}' para o tokenizer.")
else:
    print(f"Aviso: Não foi possível encontrar '{chat_template_path}'. O chat_template pode não estar configurado corretamente.")
# --- FIM DO NOVO CÓDIGO ---

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("Modelo e Tokenizer carregados e configurados para fine-tuning LoRA.")

def formatting_prompts_func(examples):
    chats = examples["messages"]
    texts = [tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False) + "<|eot_id|>" for chat in chats]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched=True)

print("\nExemplo de texto formatado para treinamento (primeiro exemplo):")
print(dataset[0]["text"])

from trl import SFTTrainer
from transformers import TrainingArguments

trainer_args = TrainingArguments(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = True, # Forçar FP16 para GPUs T4 (que não suportam BF16 nativamente)
    bf16 = False, # Desabilitar BF16 para GPUs T4
    logging_steps = 1,
    output_dir = "outputs",
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = trainer_args,
)

print("Trainer configurado. Iniciando o treinamento...")

trainer.train()

print("Treinamento concluído!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1551: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-2-7b-bnb-4bit as a legacy tokenizer.


...

Trainer configurado. Iniciando o treinamento...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 30 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 39,976,960 of 6,778,392,576 (0.59% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.374799
2,2.724398
3,2.456460
4,2.256810
5,1.931858
6,2.084815
7,1.567739
8,1.416505
9,1.277107
10,0.835643


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


Treinamento concluído!


### Entendendo e Definindo o `chat_template` para o Tokenizer

O `chat_template` é uma string que define a estrutura de um prompt de conversação que será alimentado ao modelo. Ele é crucial para modelos de chat (como o Llama-2 ou Llama-3), pois estes modelos são treinados para esperar uma formatação específica para distinguir entre falas do usuário, falas do assistente e outros elementos da conversa.

Quando você usa `tokenizer.apply_chat_template()`, o tokenizer usa esse `chat_template` para transformar uma lista de mensagens (no formato `[{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]`) em uma única string formatada que o modelo pode consumir.

**Por que é importante?**
*   **Consistência:** Garante que o modelo sempre receba os prompts no formato exato em que foi treinado, o que é vital para o desempenho.
*   **Tokens Especiais:** Inclui automaticamente tokens especiais (como `<s>`, `[INST]`, `<<SYS>>`, `<<EOT>>`, `<|begin_of_text|>`, `<|start_header_id|>`, etc.) que delimitam as diferentes partes da conversa.

**Como é definido?**
*   Muitas vezes, ele já vem pré-definido com o tokenizer ao carregar um modelo `from_pretrained` (especialmente modelos de chat).
*   Pode ser inspecionado através da propriedade `tokenizer.chat_template`.
*   Pode ser definido ou modificado manualmente, se você precisar adaptar o modelo a um novo formato de chat ou criar um template personalizado.

In [10]:
# Verificar e (opcionalmente) definir o chat_template
import os

# Verifica se o tokenizer foi carregado na célula anterior
if 'tokenizer' in locals():
    print("Tokenizador encontrado.\n")

    # 1. Inspecionar o chat_template atual
    print("Chat template atual do tokenizer:")
    if tokenizer.chat_template:
        print(tokenizer.chat_template)
    else:
        print("Nenhum chat_template definido. Tentando carregar de lora_adapters...")
        # Tenta carregar o chat_template de um arquivo se não estiver definido
        lora_model_path = "./lora_adapters"
        chat_template_path = os.path.join(lora_model_path, "chat_template.jinja")
        if os.path.exists(chat_template_path):
            with open(chat_template_path, "r", encoding="utf-8") as f:
                tokenizer.chat_template = f.read()
            print(f"Chat template carregado de '{chat_template_path}'.")
            print("Novo chat template:")
            print(tokenizer.chat_template)
        else:
            print(f"Aviso: Não foi possível encontrar '{chat_template_path}'. Definindo um template padrão para Llama-2.")
            # Fallback para um template padrão Llama-2, se o arquivo não for encontrado
            # Este é um exemplo simplificado, o template real pode ser mais complexo e inclui tokens especiais para system, user e assistant.
            # O template abaixo simula o formato ChatML para Llama-2 com system prompt dentro de INST
            tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{% if message['role'] == 'user' %}[INST] {{ message['content'] }} [/INST]{% elif message['role'] == 'system' %}[INST] <<SYS>>\n{{ message['content'] }}\n<</SYS>>\n[/INST]{% else %}{{ message['content'] }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '' }}{% endif %}"
            print("Chat template padrão Llama-2 aplicado.")

    print("\n--- Exemplo de como o template formataria uma conversa ---")
    example_chat = [
        {"role": "system", "content": "Você é um assistente útil e amigável."},
        {"role": "user", "content": "Olá, como você está?"},
        {"role": "assistant", "content": "Estou bem, obrigado! Como posso ajudar?"},
        {"role": "user", "content": "Gostaria de saber como funciona o fine-tuning de LLMs."}
    ]
    formatted_text = tokenizer.apply_chat_template(example_chat, tokenize=False, add_generation_prompt=False)
    print(formatted_text)

    # 2. Como definir ou modificar um chat_template (Exemplo para um template simples)
    # Você pode descomentar e adaptar isso se precisar de um template diferente.
    # Por exemplo, um template simplificado como o do Alpaca pode ser:
    # alpaca_template = """### Instruction:\n{system_message}\n\n### Input:\n{prompt}\n\n### Response:\n"""
    # tokenizer.chat_template = alpaca_template
    # print("\nNovo chat_template definido (exemplo Alpaca):\n" + tokenizer.chat_template)

    print("\nEntendimento do chat_template concluído.")

else:
    print("Variável 'tokenizer' não encontrada. Certifique-se de executar a célula 3.2 de fine-tuning primeiro para carregar o modelo e o tokenizer.")

Tokenizador encontrado.

Chat template atual do tokenizer:
{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{% if message['role'] == 'user' %}[INST] {{ message['content'] }} [/INST]{% elif message['role'] == 'system' %}[INST] <<SYS>>
{{ message['content'] }}
<</SYS>>
[/INST]{% else %}{{ message['content'] }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '' }}{% endif %}

--- Exemplo de como o template formataria uma conversa ---
[INST] <<SYS>>
Você é um assistente útil e amigável.
<</SYS>>
[/INST][INST] Olá, como você está? [/INST]Estou bem, obrigado! Como posso ajudar?[INST] Gostaria de saber como funciona o fine-tuning de LLMs. [/INST]

Entendimento do chat_template concluído.


### 3.3 Salvamento e Carregamento dos Pesos Ajustados

Após o fine-tuning, é crucial salvar os pesos ajustados (adaptadores LoRA) do modelo para que ele possa ser reutilizado sem a necessidade de re-treinamento. Em seguida, demonstraremos como carregar esses pesos em um novo modelo base, deixando-o pronto para inferência.

**Nota:** No Colab, os modelos são salvos no ambiente de tempo de execução. Para persistência, você precisará salvar no Google Drive ou em outro local de armazenamento.

### 3.4 Inferência com o Modelo Fine-Tuned

Com o modelo e o tokenizer carregados (e os adaptadores LoRA aplicados), podemos agora realizar inferências, ou seja, gerar respostas para novas perguntas ou prompts. É fundamental formatar o prompt de entrada de acordo com o `chat_template` esperado pelo modelo para garantir a melhor performance. O `apply_chat_template` do tokenizer é a ferramenta ideal para isso.

In [11]:
# Solução para o ImportError: Forçar a instalação de langchain_core==0.3.0
# Isso é necessário porque o `pip` pode não ter respeitado a versão fixada anteriormente
# ou outra dependência pode ter atualizado `langchain_core` para uma versão incompatível.
print("Forçando a reinstalação de langchain_core==0.3.0 para resolver incompatibilidade...")
!pip install -qqq --force-reinstall langchain_core==0.3.0
print("Verificando a versão de langchain_core após a re-instalação:")
!pip show langchain_core

print("\n======================== AVISO IMPORTANTE ========================")
print("Uma reinstalação forçada de uma biblioteca central como `langchain_core` pode exigir a")
print("**reinicialização do ambiente de execução (Runtime -> Restart runtime)** para que as alterações tenham efeito completo.")
print("Por favor, reinicie o ambiente de execução e execute as células novamente a partir da célula `7619c368`.")
print("==================================================================")

Forçando a reinstalação de langchain_core==0.3.0 para resolver incompatibilidade...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.0 which is incompatible.
langchain 1.4.0 requires langchain-core<2.0.0,>=1.6.0, but you have langchain-core 0.3.0 which is incompatible.
langchain-text-splitters 1.1.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.0 which is incompatible.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.0 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.0 which is incompatible.
langgraph-sdk 0.4.4 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.0 which is incompatible.
langchain-community 0.4.2 requires langchain-c

In [12]:
lora_model_path = "./lora_adapters"

# Salvar o modelo e tokenizer apenas se estiverem definidos na sessão atual
# Isso é útil se o kernel for reiniciado e você não quiser re-executar o treinamento
if 'model' in locals() and model is not None:
    model.save_pretrained(lora_model_path)
    print(f"Adaptadores LoRA salvos em: {lora_model_path}")
else:
    print(f"Variável 'model' não definida. Pulando o salvamento do modelo. Presume-se que o modelo já foi salvo em {lora_model_path} ou será carregado de lá.")

if 'tokenizer' in locals() and tokenizer is not None:
    tokenizer.save_pretrained(lora_model_path)
    print(f"Tokenizer salvo em: {lora_model_path}")
else:
    print(f"Variável 'tokenizer' não definida. Pulando o salvamento do tokenizer. Presume-se que o tokenizer já foi salvo em {lora_model_path} ou será carregado de lá.")


# Opcional: Empacotar e salvar no Google Drive para persistência
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r {lora_model_path} /content/drive/MyDrive/ # Copia para o Drive
# print(f"Adaptadores LoRA copiados para o Google Drive.")

# Liberar memória da GPU antes de carregar o modelo para inferência
print("\nLiberando memória da GPU (tentativa mais agressiva)...")

# Verifique e delete trainer e seu modelo interno
if 'trainer' in locals() and trainer is not None:
    if hasattr(trainer, 'model') and trainer.model is not None:
        trainer.model.to('cpu') # Move o modelo para CPU antes de deletar
        del trainer.model
    del trainer # Deletar o objeto trainer para liberar referências

# Verifique e delete model (se não foi deletado via trainer)
if 'model' in locals() and model is not None:
    model.to('cpu') # Explicitamente move para CPU
    del model

# Verifique e delete tokenizer
if 'tokenizer' in locals() and tokenizer is not None:
    del tokenizer

# Delete outras variáveis que podem consumir muita memória
if 'dataset' in locals() and dataset is not None:
    del dataset
if 'df_medical' in locals() and df_medical is not None:
    del df_medical
if 'formatted_medical_data' in locals() and formatted_medical_data is not None:
    del formatted_medical_data
# Se houver inputs/outputs de uma execução anterior de inferência, também os delete
if 'inputs' in locals() and inputs is not None:
    del inputs
if 'outputs' in locals() and outputs is not None:
    del outputs

import gc
gc.collect() # Coleta lixo do Python
torch.cuda.empty_cache() # Esvazia o cache de memória da CUDA
print("Memória da GPU liberada (agressivamente).")

# 2. Carregando os pesos ajustados para um novo modelo base
print("\nCarregando os adaptadores LoRA em um novo modelo base para inferência...")

# Recarrega o modelo base (sem LoRA)
# Importante: certifique-se de usar o mesmo modelo_name original.
new_model, new_tokenizer = FastLanguageModel.from_pretrained(
    model_name = lora_model_path, # Carrega diretamente do caminho dos adaptadores
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map = {"": 0}, # Adicionado para garantir o carregamento na GPU
)

# A linha abaixo não é mais necessária, pois FastLanguageModel.from_pretrained
# já carrega os adaptadores quando "model_name" é o caminho dos adaptadores.
# FastLanguageModel.load_adapters(new_model, lora_model_path)

print("Modelo com adaptadores LoRA carregados e pronto para inferência!")

# 3. Exemplo de Inferência com o Modelo Ajustado
print("\nRealizando uma inferência de teste com o modelo ajustado...")

# Define o prompt no formato de chat do Llama-3
# Note o 'add_generation_prompt=True' para que o modelo saiba que deve gerar uma resposta.
med_prompt = new_tokenizer.apply_chat_template([
    {"role": "user", "content": "Quais são os sintomas de um resfriado comum?"}
], tokenize=False, add_generation_prompt=True)

# Gera a resposta usando o modelo ajustado
inputs = new_tokenizer(med_prompt, return_tensors = "pt").to("cuda")

outputs = new_model.generate(**inputs, max_new_tokens = 64,
                             use_cache = True, pad_token_id = new_tokenizer.eos_token_id)

# Decodifica e imprime a resposta
response = new_tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# A resposta bruta incluirá o prompt. Vamos extrair apenas a parte da resposta do assistente.
# O Llama-3 usa <|start_header_id|>assistant<|end_header_id|> para marcar o início da resposta.
assistant_response_start = response.find("<|start_header_id|>assistant<|end_header_id|>\n")
if assistant_response_start != -1:
    clean_response = response[assistant_response_start + len("<|start_header_id|>assistant<|end_header_id|>\n"):].split("<|eot_id|>")[0].strip()
else:
    clean_response = response.strip()

print("\nResposta do modelo ajustado:")
print(clean_response)

Unsloth: Restored added_tokens_decoder metadata in ./lora_adapters/tokenizer_config.json.


Adaptadores LoRA salvos em: ./lora_adapters
Tokenizer salvo em: ./lora_adapters

Liberando memória da GPU (tentativa mais agressiva)...
Memória da GPU liberada (agressivamente).

Carregando os adaptadores LoRA em um novo modelo base para inferência...
==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load ./lora_adapters as a legacy tokenizer.
Unsloth: The tokenizer `./lora_adapters` has a {% if add_generation_prompt %} block, but it does not change the rendered output. This tokenizer was loaded from a local path. The likely cause is a downstream tool (LlamaFactory, Axolotl, etc.) that re-serialized the tokenizer during save and stripped the generation-prompt block. Either re-save with the original template, or set `tokenizer.chat_template` manually before loading. Set UNSLOTH_STRICT_CHAT_TEMPLATE=1 to raise instead of warn.
Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Modelo com adaptadores LoRA carregados e pronto para inferência!

Realizando uma inferência de teste com o modelo ajustado...

Resposta do modelo ajustado:
[INST] Quais são os sintomas de um resfriado comum? [/INST]Joelho Doendo, febre, dor de cabeça, dores musculares, tosse, dor de garganta e fadiga.<|eot_id|>|eot_id|</|eot_id|>|eot_id|</|e


## 4. Criação de Assistente Médico com LangChain

Nesta seção, construiremos um assistente médico utilizando a biblioteca LangChain. O LangChain nos permitirá integrar nosso LLM customizado, simular consultas a uma base de dados estruturada (como prontuários) e contextualizar as respostas da LLM.

### 4.1 Integração da LLM Customizada com LangChain

Para usar nosso `new_model` fine-tuned dentro do ecossistema LangChain, precisamos criar um wrapper (`LLM` customizado) que implemente a interface esperada pelo LangChain.

In [13]:
print('Reinstalando componentes do LangChain para corrigir inconsistências...')

# Reinstala as últimas versões compatíveis dos componentes do LangChain
# Usamos --force-reinstall para garantir que as versões corretas sejam carregadas
# e --no-cache-dir para garantir que não haja versões antigas em cache.
!pip install -qqq --force-reinstall --no-cache-dir langchain langchain_community langchain_core langgraph langchain-text-splitters

print('Componentes do LangChain reinstalados. Verificando versões:')
!pip show langchain
!pip show langchain_community
!pip show langchain_core
!pip show langgraph
!pip show langchain-text-splitters

print('\nSe o erro persistir, um reinício do ambiente de execução (Runtime -> Restart runtime) será necessário.')

Reinstalando componentes do LangChain para corrigir inconsistências...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.
google-adk 2.7.1 requires websockets<16,>=15.0.1, but you have websockets 16.1.1 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.
Componentes do LangChain reinstalados. Verificando versões:
Name: langchain
Version: 1.4.0
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Name: langchain-community
Ve

In [14]:
# Instalar bibliotecas adicionais necessárias para embeddings e vetor store
!pip install -qqq sentence-transformers faiss-cpu

from langchain_core.language_models import LLM
from typing import Any, List, Optional
from langchain_core.callbacks import CallbackManagerForLLMRun

class CustomFineTunedLLM(LLM):
    model: Any
    tokenizer: Any
    max_new_tokens: int = 64

    @property
    def _llm_type(self) -> str:
        return "custom_fine_tuned_llm"

    def _call(self, prompt: str, stop: Optional[List[str]] = None, run_manager: Optional[CallbackManagerForLLMRun] = None, **kwargs: Any) -> str:
        # Aplica o chat template, se necessário. Nosso new_model já espera o formato de chat
        # que foi formatado no med_prompt. Aqui, assumimos que o prompt de entrada já está no formato correto
        # para o modelo gerar a continuação.

        # Prepara o input para o modelo
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        # Gera a resposta
        outputs = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens,
                                     use_cache=True, pad_token_id=self.tokenizer.eos_token_id)

        # Decodifica a resposta
        response = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

        # Extrai apenas a parte da resposta do assistente (ignora o prompt original)
        # O formato do Llama-2/3 inclui o prompt original e o início da resposta do assistente.
        # Precisamos encontrar o marcador do assistente e retornar o texto após ele.
        assistant_prefix = "<|start_header_id|>assistant<|end_header_id|>\n"
        response_start = response.find(assistant_prefix)
        if response_start != -1:
            # Remove o prompt completo até o prefixo do assistente e qualquer token final como <|eot_id|>
            clean_response = response[response_start + len(assistant_prefix):].split("<|eot_id|>")[0].strip()
            # Para o Llama-2-7b, o template pode ser ligeiramente diferente, então ajustamos a extração
            # Se o prompt for incluído na resposta, a extração pode ser 'prompt_str + assistant_response'.
            # Vamos tentar remover o prompt de entrada da resposta gerada para obter apenas a continuação.
            if prompt in clean_response:
                clean_response = clean_response.replace(prompt, "").strip()
            elif len(inputs['input_ids'][0]) < len(outputs[0]): # Se houve geração de tokens novos
                generated_tokens = outputs[0][len(inputs['input_ids'][0]):]
                clean_response = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
            else:
                clean_response = response.strip() # Fallback
        else:
             # If assistant prefix not found, try to extract based on input prompt length
            if len(inputs['input_ids'][0]) < len(outputs[0]):
                generated_tokens = outputs[0][len(inputs['input_ids'][0]):]
                clean_response = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
            else:
                clean_response = response.strip()


        # Remove quaisquer marcadores de chat restantes no final
        clean_response = clean_response.replace("<|eot_id|>", "").strip()

        return clean_response

    @property
    def _identifying_params(self) -> dict[str, Any]:
        return {"max_new_tokens": self.max_new_tokens}

# Inicializa o LLM customizado com o modelo e tokenizer fine-tuned
# É importante que 'new_model' e 'new_tokenizer' estejam definidos a partir da célula anterior.
custom_llm = CustomFineTunedLLM(model=new_model, tokenizer=new_tokenizer, max_new_tokens=200) # Aumentar max_new_tokens para respostas mais longas

print("LLM customizado integrado com LangChain.")

# Exemplo rápido de uso do LLM customizado
print("\nTeste rápido do LLM customizado com LangChain:")
langchain_test_prompt = new_tokenizer.apply_chat_template([
    {"role": "user", "content": "Qual é o tratamento para asma?"}
], tokenize=False, add_generation_prompt=True)

response_lc = custom_llm.invoke(langchain_test_prompt)
print(f"Pergunta (LangChain): {langchain_test_prompt}")
print(f"Resposta (LangChain): {response_lc}")

Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LLM customizado integrado com LangChain.

Teste rápido do LLM customizado com LangChain:
Pergunta (LangChain): [INST] Qual é o tratamento para asma? [/INST]
Resposta (LangChain): O tratamento para a hipertensão arterial geralmente inclui mudanças no estilo de vida (dieta, exercício), e frequentemente medicamentos anti-hipertensivos prescritos por um médico.|eot_id|Para mais informações, siga o link.|eot_id|https://www. nobody-eot_id|eot_id|>www. nobody-eot_id|eot_id|>nobody|eot_id|>nobody|eot_id|>nobody</|eot_id|>|eot_id|>|eot_id|>|eot_id|>|eot_id|>|eot_id|


### 4.2 Simulação de Base de Dados Estruturadas (RAG)

Para simular a consulta a uma base de dados estruturada, utilizaremos a abordagem de Retrieval-Augmented Generation (RAG). Criaremos um pequeno conjunto de documentos que representam "prontuários e registros" e os armazenaremos em um Vector Store (FAISS) para busca de similaridade.

In [19]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # Importação corrigida para LangChain 1.x
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1. Criar dados simulados de "prontuários e registros"
# Em um cenário real, esses dados viriam de um banco de dados, PDFs, etc.
medical_records_data = [
    "Paciente João Silva, 45 anos. Histórico de asma desde a infância. Medicação atual: Salbutamol (sos), Budesonida (manutenção). Última crise em 10/05/2023, desencadeada por poeira. Aconselhado evitar alérgenos e manter inalador de resgate. Próxima consulta em 15/07/2024.",
    "Paciente Maria Souza, 60 anos. Diagnóstico de hipertensão arterial há 5 anos. Medicação: Losartana 50mg/dia. Pressão arterial média: 140/90 mmHg. Dieta com baixo teor de sódio recomendada. Exercícios leves 3 vezes por semana. Histórico familiar de doenças cardíacas.",
    "Paciente Carlos Pereira, 22 anos. Queixa principal: dor de garganta intensa, febre 38.5°C, tosse seca. Início dos sintomas há 3 dias. Exame físico: amígdalas avermelhadas. Diagnóstico provável: faringite viral. Tratamento: repouso, hidratação, analgésico (paracetamol). Reavaliar em 5 dias.",
    "Paciente Ana Costa, 30 anos. Histórico de alergia a penicilina. Apresenta erupções cutâneas e coceira generalizada após contato com produto de limpeza. Prescrito anti-histamínico e creme tópico. Aconselhado evitar o alérgeno e procurar um dermatologista se não houver melhora.",
    "Diretriz clínica para tratamento de diabetes tipo 2: Enfatiza controle glicêmico, dieta equilibrada, atividade física regular e, se necessário, medicação oral ou insulina. Monitoramento de HbA1c a cada 3-6 meses."
]

# Converte em objetos Document do LangChain
documents = [Document(page_content=record) for record in medical_records_data]

# 2. Dividir documentos em chunks (para lidar com documentos longos)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunked_documents = text_splitter.split_documents(documents)

print(f"Documentos originais: {len(documents)}")
print(f"Chunks criados: {len(chunked_documents)}")
print("\nPrimeiro chunk de exemplo:")
print(chunked_documents[0].page_content)

# 3. Escolher um modelo de embeddings leve (para rodar na T4 ou CPU se necessário)
# Usaremos um modelo da Sentence Transformers via HuggingFaceEmbeddings.
# O 'all-MiniLM-L6-v2' é um modelo pequeno e eficiente.

# Verificar se a GPU está disponível e, se sim, tentar usar para embeddings
if torch.cuda.is_available():
    print("\nUsando GPU para embeddings.")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={'device': 'cuda'})
else:
    print("\nGPU não disponível para embeddings, usando CPU.")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})


# 4. Criar um Vector Store (FAISS para busca em memória)
print("Criando FAISS vector store...")
vector_store = FAISS.from_documents(chunked_documents, embeddings)

# 5. Criar um retriever
retriever = vector_store.as_retriever()

print("Retriever para base de dados médica simulada configurado.")

# Teste do retriever
print("\nTeste do retriever para 'sintomas de asma':")
retrieved_docs = retriever.invoke("sintomas de asma")
for i, doc in enumerate(retrieved_docs):
    print(f"Documento {i+1}: {doc.page_content[:150]}...")


    !pip install --force-reinstall --no-cache-dir langchain langchain_community langchain_core langgraph langchain-text-splitters


Documentos originais: 5
Chunks criados: 5

Primeiro chunk de exemplo:
Paciente João Silva, 45 anos. Histórico de asma desde a infância. Medicação atual: Salbutamol (sos), Budesonida (manutenção). Última crise em 10/05/2023, desencadeada por poeira. Aconselhado evitar alérgenos e manter inalador de resgate. Próxima consulta em 15/07/2024.

Usando GPU para embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Criando FAISS vector store...
Retriever para base de dados médica simulada configurado.

Teste do retriever para 'sintomas de asma':
Documento 1: Paciente João Silva, 45 anos. Histórico de asma desde a infância. Medicação atual: Salbutamol (sos), Budesonida (manutenção). Última crise em 10/05/20...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 141.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 298.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 395.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 383.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 233.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 556.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 564.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 304.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.9/623.9 kB 741.2 MB/s  0:00:00
   ━━━━━━

Documento 2: Paciente Ana Costa, 30 anos. Histórico de alergia a penicilina. Apresenta erupções cutâneas e coceira generalizada após contato com produto de limpeza...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 261.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 628.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 568.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 755.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 463.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 727.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 735.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 505.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.9/623.9 kB 558.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 375.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 622.7 MB/s

Documento 3: Paciente Carlos Pereira, 22 anos. Queixa principal: dor de garganta intensa, febre 38.5°C, tosse seca. Início dos sintomas há 3 dias. Exame físico: am...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 301.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 747.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 705.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 701.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 707.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 579.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 620.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 257.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.9/623.9 kB 641.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 416.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 188.6 MB/s

Documento 4: Paciente Maria Souza, 60 anos. Diagnóstico de hipertensão arterial há 5 anos. Medicação: Losartana 50mg/dia. Pressão arterial média: 140/90 mmHg. Diet...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 248.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 700.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 590.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 588.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 603.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 653.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 739.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 451.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.9/623.9 kB 668.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 405.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 406.2 MB/s



```
# Isto está formatado como código
```

### 4.3 Construção do Pipeline LangChain para Contextualização

Agora, vamos combinar o LLM customizado e o retriever em um pipeline LangChain. Isso nos permitirá contextualizar as respostas da LLM com as informações recuperadas da nossa base de dados simulada, de forma semelhante ao funcionamento de um agente RAG.

In [20]:
import os
import sys
import site
import importlib

# Primeiro, tenta as importações padrão. Se funcionarem, ótimo.
try:
    from langchain.chains.combine_documents import create_stuff_documents_chain
    from langchain_core.prompts import ChatPromptTemplate
    from langchain.chains import create_retrieval_chain
    print("Importações do LangChain.chains realizadas com sucesso!")

except ImportError as e_standard:
    print(f"Importação padrão de LangChain.chains falhou: {e_standard}")
    print("Realizando diagnóstico aprofundado e tentando workaround...")

    # --- Início do Diagnóstico e Workaround ---
    langchain_path = None
    try:
        import langchain
        langchain_path = langchain.__path__[0]
        print(f"Pacote 'langchain' importado de: {langchain_path}")
    except ImportError:
        print("Erro: O pacote 'langchain' principal não pôde ser importado. Instalação básica falhou.")
        raise  # Re-raise se o pacote principal nem estiver lá

    chains_dir = os.path.join(langchain_path, 'chains')
    print(f"Caminho esperado para 'langchain.chains': {chains_dir}")
    chains_init_exists = os.path.exists(os.path.join(chains_dir, '__init__.py'))
    chains_dir_exists = os.path.isdir(chains_dir)

    print(f"Diretório 'chains' existe em {langchain_path}: {chains_dir_exists}")
    print(f"'__init__.py' existe em {chains_dir}: {chains_init_exists}")

    if not chains_dir_exists or not chains_init_exists:
        print("ERRO CRÍTICO: DIRETÓRIO 'langchain.chains' OU '__init__.py' AUSENTE.")
        print("A instalação do LangChain está CORROMPIDA no disco. Nenhuma solução programática pode resolver isso sem uma reinstalação completa e limpa.")
        print("A única solução *confiável* é uma REINICIALIZAÇÃO DE FÁBRICA do ambiente de execução (Runtime -> Factory reset runtime), seguida da execução de todas as células.")
        raise ImportError(f"Instalação do LangChain corrompida: módulo 'langchain.chains' não encontrado fisicamente. Detalhes: dir_exists={chains_dir_exists}, init_exists={chains_init_exists}")

    print("Diretório 'langchain.chains' e '__init__.py' encontrados fisicamente.")
    print("Parece que o problema é a resolução de módulos do Python. Tentando um workaround agressivo...")

    # Adicionar o diretório 'chains' ao sys.path para tentar forçar a resolução.
    # Isso é um hack agressivo e normalmente não é necessário para subpacotes.
    if chains_dir not in sys.path:
        sys.path.insert(0, chains_dir)
        print(f"Adicionando {chains_dir} ao sys.path para forçar a resolução.")

    # Invalida os caches do importlib para que o Python tente novamente a resolução
    importlib.invalidate_caches()

    # Tenta as importações novamente após a manipulação do sys.path e caches
    try:
        from langchain.chains.combine_documents import create_stuff_documents_chain
        from langchain_core.prompts import ChatPromptTemplate
        from langchain.chains import create_retrieval_chain
        print("Importações do LangChain.chains realizadas com sucesso após workaround de sys.path!")

    except ImportError as e_workaround:
        print(f"Importação de LangChain.chains AINDA FALHOU após workaround de sys.path: {e_workaround}")
        print("Nenhuma solução programática para este ambiente corrompido. Uma REINICIALIZAÇÃO DE FÁBRICA é estritamente necessária.")
        raise  # Re-raise se ainda falhar

    # --- Fim do Diagnóstico e Workaround ---

# 1. Definir um template de prompt para RAG (Retrieval Augmented Generation)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente médico prestativo. Use as seguintes informações de contexto para responder à pergunta do usuário. Se você não souber a resposta, diga 'Não consigo encontrar essa informação nos prontuários fornecidos.'\n\nContexto:\n{context}"),
    ("user", "{input}")
])

# 2. Criar uma cadeia para combinar os documentos (stuff documents chain)
# Esta cadeia pega os documentos recuperados e os "coloca" no prompt como contexto.
document_chain = create_stuff_documents_chain(custom_llm, rag_prompt)

# 3. Criar a cadeia de recuperação completa (retrieval chain)
# Esta cadeia combina o retriever e a document_chain.
retrieval_chain = create_retrieval_chain(retriever, document_chain)

print("Pipeline LangChain RAG configurado.")

# 4. Demonstrar uma consulta com contextualização
print("\nDemonstrando uma consulta contextualizada:")

# Pergunta que se beneficia do contexto dos prontuários
question_1 = "Qual a medicação de manutenção do paciente João Silva para asma e quando é a próxima consulta?"
response_rag_1 = retrieval_chain.invoke({"input": question_1})
print(f"Pergunta: {question_1}")
print(f"Resposta: {response_rag_1['answer']}")

print("\n--- Exemplo 2 ---")
# Pergunta que não tem resposta direta no contexto
question_2 = "Qual a dosagem recomendada de Ibuprofeno para adultos?"
response_rag_2 = retrieval_chain.invoke({"input": question_2})
print(f"Pergunta: {question_2}")
print(f"Resposta: {response_rag_2['answer']}")

Importação padrão de LangChain.chains falhou: No module named 'langchain.chains'
Realizando diagnóstico aprofundado e tentando workaround...
Pacote 'langchain' importado de: /usr/local/lib/python3.13/dist-packages/langchain
Caminho esperado para 'langchain.chains': /usr/local/lib/python3.13/dist-packages/langchain/chains
Diretório 'chains' existe em /usr/local/lib/python3.13/dist-packages/langchain: False
'__init__.py' existe em /usr/local/lib/python3.13/dist-packages/langchain/chains: False
ERRO CRÍTICO: DIRETÓRIO 'langchain.chains' OU '__init__.py' AUSENTE.
A instalação do LangChain está CORROMPIDA no disco. Nenhuma solução programática pode resolver isso sem uma reinstalação completa e limpa.
A única solução *confiável* é uma REINICIALIZAÇÃO DE FÁBRICA do ambiente de execução (Runtime -> Factory reset runtime), seguida da execução de todas as células.


ImportError: Instalação do LangChain corrompida: módulo 'langchain.chains' não encontrado fisicamente. Detalhes: dir_exists=False, init_exists=False